In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("spark-csv-data")
    .master("local[*]").config("spark.ui.port", "4040")
    .getOrCreate()
)

#
print(spark.sparkContext.uiWebUrl)

#
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/31 08:48:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://vmware-kubuntu.sandbox.net:4040


In [7]:
## By default, Spark's CSV reader treats empty strings ("") and blank values (e.g., ,, where nothing is between the commas) as null

# This code will read "N/A" as null
dfFromCSV = spark.read \
    .option("header", True) \
    .option("delimiter", ',') \
    .option("emptyValue", None) \
    .option("nullValue", None) \
    .option("treatEmptyValuesAsNulls", True) \
    .options(inferSchema=True, delimiter=',') \
    .csv("file:///apps/sandbox/datasets/employee-data/employee.csv")

#dfFromCSV.printSchema()
dfFromCSV.show(truncate=False)

+------+--------+---------+-----------+------------+----------+--------+-------+
|emp_id|emp_name|emp_role |emp_manager|emp_hiredate|emp_salary|emp_comm|dept_id|
+------+--------+---------+-----------+------------+----------+--------+-------+
|7839  |KING    |PRESIDENT|NULL       |1981-11-17  |5000      |null    |10     |
|7698  |BLAKE   |MANAGER  |7839       |1981-01-05  |2850      |125     |30     |
|7782  |CLARK   |MANAGER  |7839       |1983-09-06  |2450      |null    |10     |
|7566  |JONES   |MANAGER  |7839       |1981-02-04  |2975      |345     |20     |
|7788  |SCOTT   |ANALYST  |7566       |1984-12-17  |3000      |null    |20     |
|7902  |FORD    |ANALYST  |7566       |1981-11-27  |3000      |null    |20     |
|7369  |SMITH   |CLERK    |7902       |1980-04-04  |800       |null    |20     |
|7499  |ALLEN   |SALESMAN |7698       |1981-05-05  |1600      |300     |30     |
|7521  |WARD    |SALESMAN |7698       |1982-06-23  |1250      |500     |30     |
|7654  |MARTIN  |SALESMAN |7

In [10]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

employee_schema = StructType() \
    .add("emp_id", IntegerType(), True) \
    .add("emp_name", StringType(), True) \
    .add("emp_role", StringType(), True) \
    .add("emp_manager", StringType(), True) \
    .add("emp_hiredate", DateType(), True) \
    .add("emp_salary", IntegerType(), True) \
    .add("emp_comm", IntegerType(), True) \
    .add("dept_id", IntegerType(), True)

df_with_schema = spark.read.format("csv") \
    .option("header", True) \
    .schema(employee_schema) \
    .load("file:///apps/sandbox/datasets/employee-data/employee.csv")

#df_with_schema.printSchema()

df_with_schema.show(truncate=False)

+------+--------+---------+-----------+------------+----------+--------+-------+
|emp_id|emp_name|emp_role |emp_manager|emp_hiredate|emp_salary|emp_comm|dept_id|
+------+--------+---------+-----------+------------+----------+--------+-------+
|7839  |KING    |PRESIDENT|NULL       |1981-11-17  |5000      |NULL    |10     |
|7698  |BLAKE   |MANAGER  |7839       |1981-01-05  |2850      |125     |30     |
|7782  |CLARK   |MANAGER  |7839       |1983-09-06  |2450      |NULL    |10     |
|7566  |JONES   |MANAGER  |7839       |1981-02-04  |2975      |345     |20     |
|7788  |SCOTT   |ANALYST  |7566       |1984-12-17  |3000      |NULL    |20     |
|7902  |FORD    |ANALYST  |7566       |1981-11-27  |3000      |NULL    |20     |
|7369  |SMITH   |CLERK    |7902       |1980-04-04  |800       |NULL    |20     |
|7499  |ALLEN   |SALESMAN |7698       |1981-05-05  |1600      |300     |30     |
|7521  |WARD    |SALESMAN |7698       |1982-06-23  |1250      |500     |30     |
|7654  |MARTIN  |SALESMAN |7

In [11]:
#
## Replacing null values with some value
#
df_with_schema.fillna(value=0).show()
df_with_schema.fillna(value=0,subset=["emp_comm"]).show() # only for emp_comm column
df_with_schema.fillna(10,["emp_comm"]) \
    .fillna("---",["emp_manager"]).show()

+------+--------+---------+-----------+------------+----------+--------+-------+
|emp_id|emp_name| emp_role|emp_manager|emp_hiredate|emp_salary|emp_comm|dept_id|
+------+--------+---------+-----------+------------+----------+--------+-------+
|  7839|    KING|PRESIDENT|       NULL|  1981-11-17|      5000|       0|     10|
|  7698|   BLAKE|  MANAGER|       7839|  1981-01-05|      2850|     125|     30|
|  7782|   CLARK|  MANAGER|       7839|  1983-09-06|      2450|       0|     10|
|  7566|   JONES|  MANAGER|       7839|  1981-02-04|      2975|     345|     20|
|  7788|   SCOTT|  ANALYST|       7566|  1984-12-17|      3000|       0|     20|
|  7902|    FORD|  ANALYST|       7566|  1981-11-27|      3000|       0|     20|
|  7369|   SMITH|    CLERK|       7902|  1980-04-04|       800|       0|     20|
|  7499|   ALLEN| SALESMAN|       7698|  1981-05-05|      1600|     300|     30|
|  7521|    WARD| SALESMAN|       7698|  1982-06-23|      1250|     500|     30|
|  7654|  MARTIN| SALESMAN| 

In [18]:
from pyspark.sql.functions import *

employee_df = df_with_schema.filter(col("emp_salary") > 2000) \
.select("emp_id", "emp_name", "dept_id", "emp_salary") \
.groupby("dept_id").count() \


employee_df.show()

+-------+-----+
|dept_id|count|
+-------+-----+
|     20|    3|
|     10|    2|
|     30|    1|
+-------+-----+



In [12]:

dept_columns = ['dept_id', 'dept_name', 'dept_location']

dept_schema = StructType() \
    .add("dept_id", IntegerType(), True) \
    .add("dept_name", StringType(), True) \
    .add("dept_location", StringType(), True)

dept_df = spark.read.format("csv") \
    .option("header", True) \
    .schema(dept_schema) \
    .load("file:///apps/sandbox/datasets/employee-data/departments.csv")

#dept_df.printSchema()

dept_df.show(truncate=False)

+-------+----------+-------------+
|dept_id|dept_name |dept_location|
+-------+----------+-------------+
|10     |ACCOUNTING|NEW YORK     |
|20     |RESEARCH  |BOSTON       |
|30     |SALES     |CHICAGO      |
|40     |OPERATIONS|BOSTON       |
|50     |ADMIN     |CHICAGO      |
+-------+----------+-------------+



In [2]:
#
spark.sql("select * from csv.`file:///apps/sandbox/datasets/employee-data/departments.csv`").show()

+-------+----------+-------------+
|    _c0|       _c1|          _c2|
+-------+----------+-------------+
|dept_id| dept_name|dept_location|
|     10|ACCOUNTING|     NEW YORK|
|     20|  RESEARCH|       BOSTON|
|     30|     SALES|      CHICAGO|
|     40|OPERATIONS|       BOSTON|
|     50|     ADMIN|      CHICAGO|
+-------+----------+-------------+



In [8]:

# -- Use data source

spark.sql("""DROP TABLE departments""")

spark.sql("""
          CREATE TABLE departments (dept_id INT, dept_name STRING, dept_location STRING) 
          USING CSV 
          OPTIONS (
            path 'file:///apps/sandbox/datasets/employee-data/departments.csv',
            header true
          )
        """)


dept_df = spark.sql("select * from departments")

#
dept_df.printSchema()

dept_df.show(truncate=False)




root
 |-- dept_id: integer (nullable = true)
 |-- dept_name: string (nullable = true)
 |-- dept_location: string (nullable = true)

+-------+----------+-------------+
|dept_id|dept_name |dept_location|
+-------+----------+-------------+
|10     |ACCOUNTING|NEW YORK     |
|20     |RESEARCH  |BOSTON       |
|30     |SALES     |CHICAGO      |
|40     |OPERATIONS|BOSTON       |
|50     |ADMIN     |CHICAGO      |
+-------+----------+-------------+

